# 10 · Dateien lesen und schreiben

**Lernziele**

- Du arbeitest mit Pfaden über `pathlib`.
- Du liest und schreibst Textdateien mit `with open(...)`.
- Du verarbeitest CSV-Dateien mit dem Modul `csv`.
- Du speicherst und lädst Daten als JSON.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 7 und 14.

## Pfade mit `pathlib`

`Path` beschreibt einen Datei- oder Ordnerpfad, unabhängig vom Betriebssystem. Mit `/` setzt du Pfade zusammen.

Ein relativer Pfad wie `../daten` gilt ab dem **Arbeitsordner**. In Jupyter ist das der Ordner des Notebooks, hier `notebooks/`; `..` führt eine Ebene höher in den Projektordner, dort liegt `daten/`.

![Der relative Pfad ../daten/einkaufsliste.txt Teil für Teil: Vom Arbeitsordner notebooks/ führt .. in den Projektordner, daten in den Ordner daten/ und einkaufsliste.txt zur Datei.](../assets/10_relativer_pfad.png)

In [ ]:
from pathlib import Path

data_dir = Path("../daten")
shopping_list = data_dir / "einkaufsliste.txt"

print(shopping_list)
print(shopping_list.exists())
print(shopping_list.name, shopping_list.suffix)
print(shopping_list.resolve())  # absoluter Pfad

Ordnerinhalt auflisten:

In [ ]:
for path in sorted(data_dir.iterdir()):
    kind = "[DIR]" if path.is_dir() else f"{path.stat().st_size} Bytes"
    print(f"{path.name:<25} {kind}")

**Kurzübung 1 (Datei nicht gefunden):** Die Zelle darunter wirft `FileNotFoundError`, obwohl die Datei `faust.txt` existiert. Finde mit `Path.cwd()` heraus, wo das Notebook sucht, und baue den Pfad so, dass `exists()` `True` liefert.

In [ ]:
with open("daten/faust.txt", encoding="utf-8") as file:
    print(file.readline())

In [ ]:
print(Path.cwd())  # der Ordner dieses Notebooks
faust = Path("../daten/faust.txt")
print(faust.exists(), faust.resolve())
# Relative Pfade gelten ab dem Arbeitsordner, nicht ab dem Projektordner.
# Auch eine Chat-KI kennt den Arbeitsordner des Notebooks nicht; genau daher kommt der Fehler oft.

## Textdateien lesen

`with open(...)` öffnet die Datei und schließt sie am Ende des Blocks automatisch, auch bei Fehlern. Gib immer `encoding="utf-8"` an. Seit Python 3.15 ist UTF-8 zwar überall die Voreinstellung, ältere Versionen nehmen unter Windows aber eine andere Kodierung, und dann gehen Umlaute kaputt. *Explicit is better than implicit:* Mit `encoding="utf-8"` läuft dein Code mit jeder Version gleich.

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    content = file.read()
print(content)

Was `with` dabei Schritt für Schritt tut:

![Ablauf des with-Blocks: Datei öffnen, content = file.read() ausführen; ohne Fehler wird die Datei automatisch geschlossen und danach print(content) ausgeführt, bei einem Fehler wird sie ebenfalls automatisch geschlossen und der Fehler weitergemeldet.](../assets/10_with_block.png)

Zeile für Zeile lesen ist bei großen Dateien sparsamer:

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}: {line.strip()}")

Für kleine Dateien gibt es eine Abkürzung:

In [ ]:
shopping_list.read_text(encoding="utf-8").splitlines()

**Kurzübung 2 (Zeilenenden):** Entferne in der nummerierten Ausgabe oben `.strip()`. Welche Ausgabe erzeugt die Schleife dann, und warum? Überlege erst, dann probiere es aus. Wie bekommst du die kompakte Ausgabe auch ohne `.strip()` zurück?

In [ ]:
with open(shopping_list, encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}: {line}")
# Jede gelesene Zeile endet mit "\n", print hängt ein zweites an: Zwischen den
# Einträgen steht eine Leerzeile. Abhilfe: line.strip() entfernt das "\n" (und
# Leerzeichen am Rand), print(..., end="") lässt die Zeile unverändert:
with open(shopping_list, encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(f"{line_number}: {line}", end="")

## Textdateien schreiben

Der **Modus** bestimmt, was passiert:

| Modus | Bedeutung |
| --- | --- |
| `"r"` | lesen (Standard) |
| `"w"` | schreiben, **überschreibt** vorhandenen Inhalt |
| `"a"` | anhängen |

In [ ]:
output_dir = Path("../daten/ausgabe")
output_dir.mkdir(exist_ok=True)
notes = output_dir / "notizen.txt"

with open(notes, "w", encoding="utf-8") as file:
    file.write("Erste Zeile\n")
    file.write("Zweite Zeile\n")

with open(notes, "a", encoding="utf-8") as file:
    file.write("Angehängte Zeile\n")

print(notes.read_text(encoding="utf-8"))

Auch beim Schreiben gibt es für kleine Dateien eine Abkürzung: `write_text` schreibt den ganzen Text auf einmal und überschreibt dabei vorhandenen Inhalt, wie Modus `"w"`.

Willst du aus einer Datei lesen und in eine andere schreiben, öffnest du beide in **einem** `with`, durch Komma getrennt:

In [ ]:
greeting = output_dir / "gruss.txt"
greeting.write_text("Hallo\nTschüss\n", encoding="utf-8")

loud = output_dir / "notizen_gross.txt"
with open(notes, encoding="utf-8") as source, open(loud, "w", encoding="utf-8") as target:
    for line in source:
        target.write(line.upper())

print(greeting.read_text(encoding="utf-8"))
print(loud.read_text(encoding="utf-8"))

**Kurzübung 3 (Nur die letzte Notiz bleibt):** Der Code darunter soll drei Erinnerungen in `erinnerungen.txt` schreiben, in der Datei steht danach aber nur die letzte. Finde den Fehler und behebe ihn. Es gibt zwei Wege: Welchen nimmst du, und warum?

In [ ]:
reminders = ["Milch kaufen", "Zahnarzt anrufen", "Steuer abgeben"]
target = Path("../daten/ausgabe/erinnerungen.txt")
for reminder in reminders:
    with open(target, "w", encoding="utf-8") as file:
        file.write(reminder + "\n")
print(target.read_text(encoding="utf-8"))

In [ ]:
reminders = ["Milch kaufen", "Zahnarzt anrufen", "Steuer abgeben"]
target = Path("../daten/ausgabe/erinnerungen.txt")
with open(target, "w", encoding="utf-8") as file:  # einmal öffnen, nicht pro Erinnerung
    for reminder in reminders:
        file.write(reminder + "\n")
print(target.read_text(encoding="utf-8"))

# Zweiter Weg: in der Schleife mit Modus "a" anhängen. Dann wächst die Datei aber
# bei jedem erneuten Ausführen der Zelle um drei Zeilen. Einmal mit "w" öffnen ist
# hier richtig: Die Datei soll genau diese drei Erinnerungen enthalten.

## CSV-Dateien

CSV ist das Austauschformat für Tabellen, etwa aus Excel. Das Modul `csv` kümmert sich um Trennzeichen und Anführungszeichen. `DictReader` liefert jede Zeile als Dictionary mit den Spaltennamen als Schlüssel.

Öffne CSV-Dateien immer mit `newline=""`: Dann regelt das Modul `csv` die Zeilenenden selbst, und unter Windows entstehen beim Schreiben keine leeren Zwischenzeilen.

In [ ]:
import csv

with open(data_dir / "highscores.csv", encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        print(f"{row['name']:<10} {row['points']:>5} Punkte, Level {row['level']}")

> Werte aus CSV sind immer Strings. Zum Rechnen umwandeln: `int(row["points"])`.

Schreiben mit `DictWriter`:

In [ ]:
results = [
    {"name": "Ada", "points": 1200},
    {"name": "Linus", "points": 950},
]

with open(output_dir / "ergebnisse.csv", "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["name", "points"])
    writer.writeheader()
    writer.writerows(results)

print((output_dir / "ergebnisse.csv").read_text(encoding="utf-8"))

> **Excel und das BOM:** Excel unter Windows erkennt UTF-8 nur sicher, wenn die Datei mit einem unsichtbaren Kennzeichen beginnt, dem *BOM* (Byte Order Mark). Schreibst du für Excel, nimm `encoding="utf-8-sig"`; dann erscheinen Umlaute richtig. Liest du eine Datei aus Excel oder dem Online-Banking, nimm ebenfalls `"utf-8-sig"`. Mit `"utf-8"` bliebe das BOM als unsichtbares Zeichen vor dem ersten Spaltennamen stehen, und `row["Datum"]` fände die Spalte nicht (`KeyError`).

**Kurzübung 4 (Punkte zusammenzählen):** Berechne mit einer eigenen Schleife die Summe aller Punkte in `highscores.csv`. Welcher Fehler entsteht, wenn du `int()` vergisst?

In [ ]:
total = 0
with open(data_dir / "highscores.csv", encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        total += int(row["points"])
print(total)  # 4635

# Ohne int(): TypeError, denn 0 + "1000" verbindet eine Zahl mit einem String.
# Kürzer mit sum() und einem Generator-Ausdruck:
with open(data_dir / "highscores.csv", encoding="utf-8", newline="") as file:
    print(sum(int(row["points"]) for row in csv.DictReader(file)))

## JSON

JSON speichert verschachtelte Daten aus Listen und Dictionaries. Es ist das Standardformat von Web-APIs.

In [ ]:
import json

with open(data_dir / "personen.json", encoding="utf-8") as file:
    data = json.load(file)

for person in data["people"]:
    print(f"{person['name']} aus {person['city']}")

Zum Speichern dient `json.dump`. `indent=2` rückt die Datei lesbar ein, sonst steht alles in einer Zeile. `ensure_ascii=False` schreibt Umlaute als Umlaute, sonst steht ein Ersatzcode in der Datei. Die letzte Zeile zeigt das mit `json.dumps`, das den JSON-Text als String liefert statt ihn in eine Datei zu schreiben:

In [ ]:
data["people"].append({"name": "Grace", "city": "New York"})

with open(output_dir / "personen_neu.json", "w", encoding="utf-8") as file:
    json.dump(data, file, indent=2, ensure_ascii=False)

print((output_dir / "personen_neu.json").read_text(encoding="utf-8"))
print(json.dumps({"user": "Jörg"}))  # ohne ensure_ascii=False

**Kurzübung 5 (JSON-Rundreise):** Speichere die Einstellungen aus `settings` mit `json.dump` in die Datei `../daten/ausgabe/einstellungen.json`. Lies die Datei danach mit `json.load` wieder ein und prüfe mit `==`, ob du dasselbe Dictionary zurückbekommst.

- **Untersuchen:** Mach aus der Liste `recent_files` ein Tupel und wiederhole die Rundreise. Ist das Geladene noch gleich dem Original, und warum? Was passiert, wenn `recent_files` ein Set ist?

In [ ]:
settings = {
    "user": "Jörg",
    "font_size": 14,
    "dark_mode": True,
    "recent_files": ["bericht.docx", "umsatz.csv"],
}

In [ ]:
settings_file = output_dir / "einstellungen.json"

with open(settings_file, "w", encoding="utf-8") as file:
    json.dump(settings, file, indent=2, ensure_ascii=False)

with open(settings_file, encoding="utf-8") as file:
    loaded = json.load(file)

print(loaded)
print(loaded == settings)  # True

# Untersuchen: JSON kennt nur dict, list, str, int, float, bool und None.
# Ein Tupel speichert json.dump als Liste, zurück kommt also eine Liste.
with_tuple = settings.copy()
with_tuple["recent_files"] = ("bericht.docx", "umsatz.csv")
with open(settings_file, "w", encoding="utf-8") as file:
    json.dump(with_tuple, file, indent=2, ensure_ascii=False)
with open(settings_file, encoding="utf-8") as file:
    loaded = json.load(file)
print(loaded["recent_files"], loaded == with_tuple)  # Liste, False

# Ein Set kennt JSON gar nicht: TypeError (Zelle darunter).
# Abhilfe: das Set vor dem Speichern mit sorted(...) in eine Liste umwandeln.

In [ ]:
with_set = settings.copy()
with_set["recent_files"] = {"bericht.docx", "umsatz.csv"}
json.dumps(with_set)  # wie json.dump, liefert den Text aber ohne Datei: kein halb geschriebenes JSON

## Zusammenfassung

- Relative Pfade gelten ab dem Arbeitsordner, in Jupyter dem Ordner des Notebooks.
- `Path("ordner") / "datei.txt"` baut Pfade, `.exists()`, `.iterdir()`, `.mkdir()` helfen beim Umgang.
- Immer `with open(pfad, modus, encoding="utf-8") as file:`; zwei Dateien in einem `with`, durch Komma getrennt.
- Modus `"r"` lesen, `"w"` überschreiben, `"a"` anhängen. Kleine Dateien: `read_text()` und `write_text()`.
- CSV: `csv.DictReader` und `csv.DictWriter`, dabei `newline=""` angeben. Für Excel: `encoding="utf-8-sig"`.
- JSON: `json.load` / `json.dump(..., indent=2, ensure_ascii=False)`.